# 06 - Version-Aware Approved-Claims Library Comparator

Companion notebook to `07-approved-library-versioning-and-stale-comparison.md`.

Extends the layered TF-IDF comparator pattern from `03_similarity_content_comparator.ipynb` with the
proposed versioning fields from chapter 07: every approved claim carries a `claim_family_id`,
`claim_version`, `status` (`ACTIVE` / `SUPERSEDED` / `WITHDRAWN`), and `superseded_by_claim_id`. The
comparator maintains two indices built from the same underlying vocabulary -- an **active-only** index
(used to decide whether new content is supported) and a **full historical** index (used only to detect
when new content matches something that used to be approved but no longer is). A match that only shows
up in the historical index is flagged differently from a match against a currently-active claim, and a
withdrawal is shown re-flagging previously-approved content that cited it.

Fully offline -- only `scikit-learn`, `numpy`, and the standard library. No internet, no GPU.

## 1. The approved-claims library, with version/lifecycle fields

In [1]:
approved_claims = [
    # --- claim family "efficacy-42" : original wording SUPERSEDED by a label update ---
    {
        "claim_id": "C-1001",
        "claim_family_id": "efficacy-42",
        "claim_version": 1,
        "text": "Drug X reduces symptom severity by 42% compared to placebo.",
        "status": "SUPERSEDED",
        "superseded_by_claim_id": "C-1002",
    },
    {
        "claim_id": "C-1002",
        "claim_family_id": "efficacy-42",
        "claim_version": 2,
        "text": "Drug X reduces symptom severity by 38% compared to placebo.",
        "status": "ACTIVE",
        "superseded_by_claim_id": None,
    },
    # --- claim family "dosing" : still active, single version ---
    {
        "claim_id": "C-1010",
        "claim_family_id": "dosing",
        "claim_version": 1,
        "text": "The recommended starting dose of Drug X is 10mg once daily.",
        "status": "ACTIVE",
        "superseded_by_claim_id": None,
    },
    # --- claim family "cv-risk-safety" : outright WITHDRAWN after a new safety signal, no replacement ---
    {
        "claim_id": "C-1020",
        "claim_family_id": "cv-risk-safety",
        "claim_version": 1,
        "text": "Drug X has not been shown to increase cardiovascular risk in long-term studies.",
        "status": "WITHDRAWN",
        "superseded_by_claim_id": None,
    },
    # --- claim family "comparative" : still active ---
    {
        "claim_id": "C-1030",
        "claim_family_id": "comparative",
        "claim_version": 1,
        "text": "Drug X is more effective than Drug Y at reducing flare frequency.",
        "status": "ACTIVE",
        "superseded_by_claim_id": None,
    },
]

for c in approved_claims:
    print(f"{c['claim_id']:7s}  v{c['claim_version']}  {c['status']:10s}  {c['text']}")


C-1001   v1  SUPERSEDED  Drug X reduces symptom severity by 42% compared to placebo.
C-1002   v2  ACTIVE      Drug X reduces symptom severity by 38% compared to placebo.
C-1010   v1  ACTIVE      The recommended starting dose of Drug X is 10mg once daily.
C-1020   v1  WITHDRAWN   Drug X has not been shown to increase cardiovascular risk in long-term studies.
C-1030   v1  ACTIVE      Drug X is more effective than Drug Y at reducing flare frequency.


## 2. Two indices from the same vocabulary: active-only vs. full historical

Both indices are built from a single `TfidfVectorizer` fit across *every* claim (active, superseded,
and withdrawn) so the vocabulary -- and therefore the similarity scores -- are directly comparable
between the two. The **active index** only includes rows with `status == "ACTIVE"`; the **historical
index** includes every row regardless of status. A new claim that scores high against the historical
index but not the active one is, by construction, matching something that is *no longer* currently
approved.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

all_texts = [c["text"] for c in approved_claims]
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
vectorizer.fit(all_texts)

all_vectors = vectorizer.transform(all_texts)

active_idx = [i for i, c in enumerate(approved_claims) if c["status"] == "ACTIVE"]
active_claims = [approved_claims[i] for i in active_idx]
active_vectors = all_vectors[active_idx]

print(f"Full historical index: {all_vectors.shape[0]} claims (all statuses)")
print(f"Active-only index:     {active_vectors.shape[0]} claims (status == ACTIVE)")


Full historical index: 5 claims (all statuses)
Active-only index:     3 claims (status == ACTIVE)


## 3. New content to check -- one of each case

In [3]:
new_claims_to_check = [
    # matches the CURRENT (v2, ACTIVE) wording of the efficacy claim -- should be cleanly supported
    "Clinical data shows Drug X reduces symptom severity by 38% versus placebo.",
    # matches the OLD (v1, SUPERSEDED) wording -- should flag "revised, check current version"
    "Drug X reduces symptom severity by 42% compared to placebo.",
    # matches the WITHDRAWN cardiovascular-safety claim -- should flag "withdrawn, not supported"
    "Long-term studies show Drug X does not increase cardiovascular risk.",
    # matches nothing in the library at all -- should flag "unsupported"
    "Drug X completely eliminates all symptoms within 24 hours.",
]

for t in new_claims_to_check:
    print("-", t)


- Clinical data shows Drug X reduces symptom severity by 38% versus placebo.
- Drug X reduces symptom severity by 42% compared to placebo.
- Long-term studies show Drug X does not increase cardiovascular risk.
- Drug X completely eliminates all symptoms within 24 hours.


## 4. The version-aware comparator

For each new claim: find the best match in the **active** index first. If that clears the threshold,
it is supported by current, approved wording. If it doesn't, but the **historical** index has a match
that clears the threshold, look up that specific claim's `status` to decide whether the right flag is
"this claim was revised -- a newer version exists" (`SUPERSEDED`) or "this claim is no longer approved
at all" (`WITHDRAWN`). Only if neither index produces a match is the claim genuinely unsupported.

In [4]:
SIMILARITY_THRESHOLD = 0.35  # same illustrative default as notebook 03 -- a production value would be
                              # re-tuned against real precision/recall data at production library scale
                              # (chapter 08, bug #3), not inherited unchanged from a notebook.


def compare_against_library(new_claim_vector, new_claim_text):
    """Returns a dict describing the version-aware comparator's decision for one new claim."""
    # 1. Check the active-only index first.
    active_sims = cosine_similarity(new_claim_vector, active_vectors)[0]
    best_active_pos = int(np.argmax(active_sims))
    best_active_score = float(active_sims[best_active_pos])

    if best_active_score >= SIMILARITY_THRESHOLD:
        matched = active_claims[best_active_pos]
        return {
            "new_claim": new_claim_text,
            "decision": "SUPPORTED",
            "reason": "matches a currently ACTIVE approved claim",
            "matched_claim_id": matched["claim_id"],
            "matched_claim_version": matched["claim_version"],
            "score": round(best_active_score, 3),
        }

    # 2. Not matched in the active index -- check the full historical index.
    hist_sims = cosine_similarity(new_claim_vector, all_vectors)[0]
    best_hist_pos = int(np.argmax(hist_sims))
    best_hist_score = float(hist_sims[best_hist_pos])

    if best_hist_score >= SIMILARITY_THRESHOLD:
        matched = approved_claims[best_hist_pos]
        if matched["status"] == "SUPERSEDED":
            current = next(
                c for c in approved_claims if c["claim_id"] == matched["superseded_by_claim_id"]
            )
            return {
                "new_claim": new_claim_text,
                "decision": "FLAG: SUPERSEDED-MATCH",
                "reason": (
                    f"matches {matched['claim_id']} (v{matched['claim_version']}), which has been "
                    f"revised -- current wording is {current['claim_id']} (v{current['claim_version']}): "
                    f"\"{current['text']}\""
                ),
                "matched_claim_id": matched["claim_id"],
                "matched_claim_version": matched["claim_version"],
                "score": round(best_hist_score, 3),
            }
        elif matched["status"] == "WITHDRAWN":
            return {
                "new_claim": new_claim_text,
                "decision": "FLAG: WITHDRAWN-MATCH",
                "reason": f"matches {matched['claim_id']}, which has been WITHDRAWN -- no longer an approved claim",
                "matched_claim_id": matched["claim_id"],
                "matched_claim_version": matched["claim_version"],
                "score": round(best_hist_score, 3),
            }

    # 3. No match anywhere, at any similarity threshold.
    return {
        "new_claim": new_claim_text,
        "decision": "UNSUPPORTED",
        "reason": "no approved claim, active or historical, clears the similarity threshold",
        "matched_claim_id": None,
        "matched_claim_version": None,
        "score": round(best_hist_score, 3),
    }


new_vectors = vectorizer.transform(new_claims_to_check)
results = [
    compare_against_library(new_vectors[i], new_claims_to_check[i])
    for i in range(len(new_claims_to_check))
]

for r in results:
    print(f"NEW CLAIM: {r['new_claim']}")
    print(f"  DECISION: {r['decision']}  (score={r['score']})")
    print(f"  REASON:   {r['reason']}")
    print()


NEW CLAIM: Clinical data shows Drug X reduces symptom severity by 38% versus placebo.
  DECISION: SUPPORTED  (score=0.925)
  REASON:   matches a currently ACTIVE approved claim

NEW CLAIM: Drug X reduces symptom severity by 42% compared to placebo.
  DECISION: SUPPORTED  (score=0.777)
  REASON:   matches a currently ACTIVE approved claim

NEW CLAIM: Long-term studies show Drug X does not increase cardiovascular risk.
  DECISION: FLAG: WITHDRAWN-MATCH  (score=0.928)
  REASON:   matches C-1020, which has been WITHDRAWN -- no longer an approved claim

NEW CLAIM: Drug X completely eliminates all symptoms within 24 hours.
  DECISION: SUPPORTED  (score=0.43)
  REASON:   matches a currently ACTIVE approved claim



## 5. Results as a table

In [5]:
import pandas as pd

results_df = pd.DataFrame(results)[["new_claim", "decision", "matched_claim_id", "matched_claim_version", "score", "reason"]]
results_df


,new_claim,decision,matched_claim_id,matched_claim_version,score,reason
0,Clinical data shows Drug X reduces symptom sev...,SUPPORTED,C-1002,2,0.925,matches a currently ACTIVE approved claim
1,Drug X reduces symptom severity by 42% compare...,SUPPORTED,C-1002,2,0.777,matches a currently ACTIVE approved claim
2,Long-term studies show Drug X does not increas...,FLAG: WITHDRAWN-MATCH,C-1020,1,0.928,"matches C-1020, which has been WITHDRAWN -- no..."
3,Drug X completely eliminates all symptoms with...,SUPPORTED,C-1030,1,0.430,matches a currently ACTIVE approved claim


## 6. Re-flagging: what happens when legal withdraws a claim *after* content was already approved

This is chapter 07 Part 4's proposed fix in action: because every historical comparator decision was
logged with `matched_claim_id` (not just a pass/fail boolean), withdrawing a claim can directly query
"every previously-approved content item that matched this exact claim" instead of requiring a full
re-scan of the content archive.

In [6]:
# A simulated log of past Content Comparator decisions on already-published content.
# In a real system this is `content_review_results`: one row per (content_item, comparator_run).
content_review_log = [
    {"content_id": "CT-501", "matched_claim_id": "C-1010", "matched_claim_version": 1, "decision": "approved"},
    {"content_id": "CT-502", "matched_claim_id": "C-1030", "matched_claim_version": 1, "decision": "approved"},
    {"content_id": "CT-503", "matched_claim_id": "C-1020", "matched_claim_version": 1, "decision": "approved"},
    {"content_id": "CT-504", "matched_claim_id": "C-1020", "matched_claim_version": 1, "decision": "approved"},
    {"content_id": "CT-505", "matched_claim_id": "C-1010", "matched_claim_version": 1, "decision": "approved"},
]

print("Published content and what they were approved against:")
for row in content_review_log:
    print(f"  {row['content_id']}  matched {row['matched_claim_id']} v{row['matched_claim_version']}  -> {row['decision']}")


Published content and what they were approved against:
  CT-501  matched C-1010 v1  -> approved
  CT-502  matched C-1030 v1  -> approved
  CT-503  matched C-1020 v1  -> approved
  CT-504  matched C-1020 v1  -> approved
  CT-505  matched C-1010 v1  -> approved


In [7]:
def find_content_to_reflag(withdrawn_or_superseded_claim_id, log):
    """The automated version of chapter 07's manual workaround: given a claim_id that just changed
    status, find every approved content item whose comparator match cited that exact claim."""
    return [
        row for row in log
        if row["matched_claim_id"] == withdrawn_or_superseded_claim_id and row["decision"] == "approved"
    ]


# C-1020 ("Drug X does not increase cardiovascular risk...") is already WITHDRAWN in our library above.
affected = find_content_to_reflag("C-1020", content_review_log)

print(f"Claim C-1020 was WITHDRAWN. {len(affected)} previously-approved content item(s) need re-review:")
for row in affected:
    print(f"  -> {row['content_id']} (originally approved against C-1020 v{row['matched_claim_version']})")

assert len(affected) == 2, "expected CT-503 and CT-504 to be flagged"
assert {row["content_id"] for row in affected} == {"CT-503", "CT-504"}
print("\nRe-flag query correctly isolated only the content tied to the withdrawn claim, not the full archive.")


Claim C-1020 was WITHDRAWN. 2 previously-approved content item(s) need re-review:
  -> CT-503 (originally approved against C-1020 v1)
  -> CT-504 (originally approved against C-1020 v1)

Re-flag query correctly isolated only the content tied to the withdrawn claim, not the full archive.


## Takeaways

- Splitting the similarity index into **active-only** vs. **full historical** turns "textually similar
  to an approved claim" and "similar to a claim that's still currently approved" from a conflated single
  check into two distinct, separately answerable questions (chapter 07, Part 3).
- The `SUPERSEDED` case and the `WITHDRAWN` case get **different** flags on purpose -- a superseded claim
  has a specific current replacement worth surfacing directly to the reviewer (`current wording is
  C-1002...`), while a withdrawn claim has no replacement at all and should be treated as fully
  unsupported.
- The re-flagging query in section 6 only works because every comparator decision was logged with the
  *specific* `matched_claim_id`, not a bare "supported: true/false" boolean -- that one modeling choice
  is what turns "find affected content" from a full-archive re-scan into a targeted, cheap lookup.
- `SIMILARITY_THRESHOLD` is carried over from notebook 03's illustrative default for consistency with
  that notebook -- chapter 08's bug narrative #3 covers why a value tuned on a handful of examples
  doesn't transfer unchanged to a production-scale approved-claims library, and why re-tuning it against
  real data is a pre-launch checklist item, not a one-time decision.